# VQA Mutual Cognition Experiment -- Qwen2.5-VL-3B

Agent A sees image -> describes in 3 phrases (token-budgeted)  
Agent B sees description + question + choices -> answers (A/B/C/D)  
4 conditions: blind / a_aware / b_aware / mutual  
Token budgets: [16, 24, 32, 48]  
30 questions from ScienceQA (image subset), seed=42, A caching

In [1]:
# ============================================================
# Cell 1: Install + Load Qwen2.5-VL-3B
# ============================================================
!pip install -q torch transformers accelerate qwen-vl-utils
!pip install -q datasets  # for ScienceQA

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch

print("Loading Qwen2.5-VL-3B-Instruct...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype="auto",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
print(f"Model loaded. Device: {model.device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 64.1 MB/s eta 0:00:00:00:0100:01
Loading Qwen2.5-VL-3B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded. Device: cuda:0


In [3]:
# ============================================================
# Cell 2: ScienceQA Loader -- filter image questions
# ============================================================
import random, re, time, json

from datasets import load_dataset

print("Loading ScienceQA dataset (test split)...")
ds = load_dataset("derek-thomas/ScienceQA", split="test")
print(f"Loaded {len(ds)} rows")

# Filter for questions with images and >= 2 choices
image_rows = []
for i, row in enumerate(ds):
    img = row.get("image", None)
    if img is not None:
        choices = row.get("choices", [])
        if len(choices) >= 2:
            image_rows.append(row)

print(f"Found {len(image_rows)} questions with images")

# Sample 30
random.seed(42)
samples = random.sample(image_rows, min(10, len(image_rows)))
print(f"Sampled {len(samples)} questions")

# Format
LETTERS = "ABCDEFGH"

def format_question(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{LETTERS[i]}) {c}" for i, c in enumerate(choices))
    answer_idx = row["answer"]  # integer index
    answer_letter = LETTERS[answer_idx]
    return {
        "image": row["image"],      # PIL Image
        "question": row["question"],
        "choices_text": choices_text,
        "answer": answer_letter,
        "n_choices": len(choices),
    }

questions = [format_question(s) for s in samples]
print(f"Ready: {len(questions)} formatted questions")
print(f"Example: {questions[0]['question'][:80]}... Answer={questions[0]['answer']}")

Loading ScienceQA dataset (test split)...
Loaded 4241 rows
Found 2017 questions with images
Sampled 10 questions
Ready: 10 formatted questions
Example: Think about the magnetic force between the magnets in each pair. Which of the fo... Answer=B


In [ ]:
# ============================================================
# Cell 3: VQA 4조건 (B_AWARE를 A 상태별로 분리)
# ============================================================

A_BLIND = """You are Agent A.

You can inspect the image, but:
- reliable for large objects, regions, and spatial relations
- may miss small details, text, or fine distinctions
- report only visible evidence; do not guess

Output exactly 4 short lines:

Type: (photo, map, diagram, chart, or illustration)
Key: main visible objects, regions, or entities
Relation: main spatial relation, interaction, or directional pattern
Limit: one important unclear, small, or unreadable detail

Rules:
- be concise and factual
- do not answer the question
- do not infer unseen details"""

A_AWARE = """You are Agent A.

You can inspect the image, but:
- reliable for large objects, regions, and spatial relations
- may miss small details, text, or fine distinctions
- report only visible evidence; do not guess

You also know the question and answer choices.

Output exactly 4 short lines:

Type: (photo, map, diagram, chart, or illustration)
Key: the visible evidence most useful for distinguishing the answer choices
Relation: main spatial relation, interaction, or directional pattern
Limit: one important ambiguity, missing fine detail, or unreadable element

Rules:
- be concise and factual
- Key must focus on evidence that separates the answer choices
- explicitly note if a crucial detail is too small or unclear
- do NOT mention answer letters
- do NOT copy answer choices"""

B_BLIND = """You are Agent B.

You are given:
- a report from Agent A
- a question
- answer choices

Agent A saw the image but did NOT know the question or answer choices.

Interpret the report accordingly:
- mentioned details are likely salient but not task-targeted
- missing small details are absent
- treat the report as a general visual summary, not a task-optimized one

Output ONLY: A, B, C, or D."""

B_AWARE_FOR_A_BLIND = """You are Agent B.

You are given:
- a report from Agent A
- a question
- answer choices

About Agent A:
- A saw the image but did NOT know the question or answer choices
- A is reliable for large objects, actions, and spatial relations
- A may miss small objects, text, and subtle details

Interpret the report accordingly:
- mentioned details are likely salient but not task-targeted
- missing small details are absent
- treat the report as a general visual summary, not a task-optimized one

Output ONLY: A, B, C, or D."""

B_AWARE_FOR_A_AWARE = """You are Agent B.

You are given a description from Agent A, a question, and answer choices.

About Agent A:
- A saw the image and DID know the question and answer choices
- A is reliable for salient objects and spatial relations
- A may still miss small details, text, and subtle distinctions
- because A knew the task, mentioned details are likely chosen for their usefulness in distinguishing the answers

Use the description accordingly:
- trust explicitly mentioned details strongly
- treat mentioned cues as likely diagnostic for the question
- still do not assume every omitted fine detail is absent

Output ONLY: A, B, C, or D."""

# ── Conditions (B varies by A's state) ────────────────────────
CONDITIONS = {
    "blind":   {"a_type": "blind",  "b_prompt": "B_BLIND"},
    "a_aware": {"a_type": "aware",  "b_prompt": "B_BLIND"},
    "b_aware": {"a_type": "blind",  "b_prompt": "B_AWARE_FOR_A_BLIND"},
    "mutual":  {"a_type": "aware",  "b_prompt": "B_AWARE_FOR_A_AWARE"},
}

B_PROMPTS = {
    "B_BLIND": B_BLIND,
    "B_AWARE_FOR_A_BLIND": B_AWARE_FOR_A_BLIND,
    "B_AWARE_FOR_A_AWARE": B_AWARE_FOR_A_AWARE,
}

TX_BUDGETS = [16, 24, 32, 48]

# ── Answer extraction ────────────────────────────────────────
def extract_answer(resp, n_choices=4):
    valid = set(LETTERS[:n_choices])
    t = resp.strip().upper()
    if len(t) == 1 and t in valid:
        return t
    for pat in [
        r'(?:answer|choice)[\s:is]*([A-' + LETTERS[n_choices-1] + r'])\b',
        r'^([A-' + LETTERS[n_choices-1] + r'])[)\.]',
        r'\b([A-' + LETTERS[n_choices-1] + r'])\b',
    ]:
        m = re.search(pat, t, re.IGNORECASE | re.MULTILINE)
        if m and m.group(1).upper() in valid:
            return m.group(1).upper()
    return "N/A"

# ── Inference helpers ─────────────────────────────────────────
def agent_a_vision(system_prompt, pil_image, user_text, max_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": [
            {"type": "image", "image": pil_image},
            {"type": "text", "text": user_text},
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[pil_image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()

def agent_b_text(system_prompt, user_text, max_tokens=16):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor.tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()

# ── Main loop ─────────────────────────────────────────────────
results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}
call_count = 0
t_start = time.time()

print(f"VQA Mutual Cognition (B_AWARE split by A state)")
print(f"  Questions: {len(questions)}, Budgets: {TX_BUDGETS}")
print()

for qi, q in enumerate(questions):
    print(f"\n{'='*60}")
    print(f"Q{qi+1}/{len(questions)}: {q['question']}")
    print(f"Expected: {q['answer']}")
    print(f"Choices:\n{q['choices_text']}")
    print(f"{'='*60}")

    for budget in TX_BUDGETS:
        a_cache = {}

        for cond_name, cond in CONDITIONS.items():
            a_type = cond["a_type"]
            b_prompt_key = cond["b_prompt"]

            # Agent A (cached)
            if a_type not in a_cache:
                if a_type == "blind":
                    a_desc = agent_a_vision(A_BLIND, q["image"], "Describe the image.", max_tokens=budget)
                else:
                    a_desc = agent_a_vision(A_AWARE, q["image"],
                        f"Question: {q['question']}\nChoices:\n{q['choices_text']}\n\nDescribe the image.",
                        max_tokens=budget)
                a_cache[a_type] = a_desc
                call_count += 1
                print(f"\n  [A-{a_type} @{budget}tok]:")
                print(f"    {a_desc}")

            a_desc = a_cache[a_type]

            # Agent B
            b_system = B_PROMPTS[b_prompt_key]
            b_input = f"Description:\n{a_desc}\n\nQuestion: {q['question']}\nChoices:\n{q['choices_text']}\n\nAnswer:"
            b_resp = agent_b_text(b_system, b_input, max_tokens=12)
            call_count += 1

            pred = extract_answer(b_resp, q["n_choices"])
            correct = int(pred == q["answer"])
            results[budget][cond_name].append(correct)

            mark = "\u2713" if correct else "\u2717"
            print(f"  [{cond_name:8s}] B_raw='{b_resp}' -> pred={pred} {mark} (expected {q['answer']})")

        # Budget summary
        accs = {c: sum(results[budget][c]) for c in CONDITIONS}
        print(f"\n  --- @{budget}tok summary: " + " | ".join(f"{c}:{accs[c]}/{qi+1}" for c in CONDITIONS))

    elapsed = time.time() - t_start
    eta = elapsed / (qi+1) * (len(questions)-qi-1)
    print(f"\n  [{call_count} calls, {elapsed:.0f}s, ETA {eta:.0f}s]", flush=True)

# ── Results ────────────────────────────────────────────────────
total_time = time.time() - t_start
print(f"\n{'='*60}")
print(f"RESULTS -- {len(questions)} questions, {call_count} calls, {total_time:.0f}s")
print(f"{'='*60}\n")

print(f"{'Budget':<10}" + "".join(f"{c:<12}" for c in CONDITIONS))
print("-" * 58)
for budget in TX_BUDGETS:
    row = f"{budget}tok     "
    for c in CONDITIONS:
        vals = results[budget][c]
        acc = sum(vals)/len(vals)*100 if vals else 0
        row += f"{acc:>5.1f}%      "
    print(row)

print(f"\n--- 2x2 Interaction ---")
for budget in TX_BUDGETS:
    bl = sum(results[budget]["blind"])/len(results[budget]["blind"])*100
    aa = sum(results[budget]["a_aware"])/len(results[budget]["a_aware"])*100
    ba = sum(results[budget]["b_aware"])/len(results[budget]["b_aware"])*100
    mu = sum(results[budget]["mutual"])/len(results[budget]["mutual"])*100
    a_eff = ((aa+mu)/2) - ((bl+ba)/2)
    b_eff = ((ba+mu)/2) - ((bl+aa)/2)
    interaction = mu - aa - ba + bl
    print(f"  @{budget}tok: A={a_eff:+.1f}pp B={b_eff:+.1f}pp Int={interaction:+.1f}pp | mutual={mu:.0f}% blind={bl:.0f}%")